# 03 – XGBoost Training, Evaluation & Deployment Preparation
End-to-end training notebook for the final production model.

## Objectives
- Train the final XGBoost model
- Tune hyperparameters
- Evaluate performance
- Analyze errors
- Export a deployment-ready pipeline

In [16]:
import pandas as pd
import numpy as np
import joblib
import json 

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor

import matplotlib.pyplot as plt


In [17]:
df_model = pd.read_parquet("../data/model_ready.parquet", engine="pyarrow")

df_model.dtypes

price             float64
miles             float64
year              float64
make                  str
body_type             str
vehicle_type          str
drivetrain            str
transmission          str
fuel_type             str
engine_size       float64
engine_block          str
city                  str
province              str
miles_was_zero      int64
car_age           float64
miles_per_year    float64
model_trim            str
region                str
dtype: object

Pasting code from N02 -> Preprocessing Pipelines, defining X and y, target and predictor variables as well Column transformer

In [18]:
TARGET = "price"

X = df_model.drop(columns=[TARGET]).copy()
y = df_model[TARGET].copy()

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42)

In [20]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()


In [21]:
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

X[categorical_features].nunique().sort_values(ascending=False)

model_trim      5998
region           963
city             758
make              57
fuel_type         24
body_type         21
province          15
engine_block       3
drivetrain         3
vehicle_type       2
transmission       2
dtype: int64

In [22]:
high_cardinality_features = [
    "model_trim",
    "region",
    "city"
]

In [23]:
cat_columns_to_drop = [
    'model_trim' , 'region' , 'city'
]

low_cardinality_features = [
    col for col in categorical_cols
    if col not in cat_columns_to_drop
]

# High-cardinality categorical columns with separate transformers
model_trim_features = ["model_trim"]
region_features = ["region"]
city_features = ["city"]

# Final selected features
selected_features = (
    numeric_cols
    + low_cardinality_features
    + model_trim_features
    + region_features
    + city_features
)

selected_features = list(dict.fromkeys(selected_features))

# Remove accidental duplicates
selected_features = list(dict.fromkeys(selected_features))

In [24]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]
)

low_cardinality_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

#
model_trim_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ))
])

region_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

city_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

preprocessor = ColumnTransformer(transformers=[
    ("numerical", numeric_transformer, numeric_cols),

    ("low_cardinality_categorical",
     low_cardinality_categorical_transformer,
     low_cardinality_features),

    ("model_trim",
     model_trim_transformer,
     model_trim_features),

    ("region",
     region_transformer,
     region_features),

    ("city",
     city_transformer,
     city_features)
])

preprocessor 

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('low_cardinality_categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_

## Preprocessing pipeline
Defined inline rather than loaded from a fitted object saved in Notebook 02.

High-cardinality columns (model_trim, region, city) use one-hot encoding based on frequency to keep the matrix narrow and reduce overfitting on rare categories.

In [25]:
def evaluate(model, X, y):
    pred = model.predict(X)
    return {

        "MAE" : mean_absolute_error(y, pred),
        "R2" : r2_score(y, pred),
        "RMSE" : np.sqrt(mean_squared_error(y, pred)), 

    }

baseline_xgb = XGBRegressor(
        n_estimators=1500,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=1,
    gamma=0,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1,
    reg_alpha=0,
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
)

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline_xgb)
])

baseline_pipeline.fit(X_train, y_train)
baseline_metrics = evaluate(baseline_pipeline, X_test, y_test)
print("BASELINE (beat this):", baseline_metrics)
BASELINE_MAE = baseline_metrics["MAE"]

BASELINE (beat this): {'MAE': 2738.722653593595, 'R2': 0.9356699216410026, 'RMSE': np.float64(5140.382981240645)}


## Hyperparameter tuning
Staged GridSearch (coordinate descent) rather than one combinatorial grid: tree complexity → regularization → learning rate/estimators. Each stage's winners are fixed for the next

In [26]:
from sklearn.model_selection import GridSearchCV


CV_FOLDS = 3
SCORING = "neg_mean_absolute_error"

def make_pipeline(**xgb_overrides):

    params = dict(
        objective="reg:squarederror",
        eval_metric="mae",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1,
    )
    params.update(xgb_overrides)
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(**params)),
    ])

def run_grid(param_grid, fixed_params, stage_name):
   
    pipe = make_pipeline(**fixed_params)
    grid = {f"model__{k}": v for k, v in param_grid.items()}
    search = GridSearchCV(
        pipe,
        param_grid=grid,
        scoring=SCORING,
        cv=CV_FOLDS,
        n_jobs=1,        
        verbose=2,
        refit=True,
    )
    print(f"=== {stage_name} ===")
    search.fit(X_train, y_train)
    print("Best CV MAE:", -search.best_score_)
    print("Best params:", search.best_params_)
    return search


### Stage A — tree complexity

Searches `max_depth`, `min_child_weight`, `gamma`, `subsample`, `colsample_bytree` with `learning_rate`/`n_estimators` held at a moderate anchor value (not final — those get tuned in Stage C).

Grid size: 3 × 2 × 2 × 2 × 2 = 48 combinations × `CV_FOLDS=3` = **144 fits**.

In [27]:
# Anchor learning_rate/n_estimators while we search tree-complexity params.
# These are NOT the final values - Stage C tunes them properly.
STAGE_A_ANCHOR = dict(learning_rate=0.05, n_estimators=400)

stage_a_grid = {
    "max_depth": [6, 8, 10],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

search_a = run_grid(stage_a_grid, STAGE_A_ANCHOR, "Stage A: tree complexity")
stage_a_best = {k.replace("model__", ""): v for k, v in search_a.best_params_.items()}
stage_a_best

=== Stage A: tree complexity ===
Fitting 3 folds for each of 48 candidates, totalling 144 fits
[CV] END model__colsample_bytree=0.8, model__gamma=0, model__max_depth=6, model__min_child_weight=1, model__subsample=0.8; total time=   5.4s
[CV] END model__colsample_bytree=0.8, model__gamma=0, model__max_depth=6, model__min_child_weight=1, model__subsample=0.8; total time=   5.3s
[CV] END model__colsample_bytree=0.8, model__gamma=0, model__max_depth=6, model__min_child_weight=1, model__subsample=0.8; total time=   5.1s
[CV] END model__colsample_bytree=0.8, model__gamma=0, model__max_depth=6, model__min_child_weight=1, model__subsample=1.0; total time=   5.2s
[CV] END model__colsample_bytree=0.8, model__gamma=0, model__max_depth=6, model__min_child_weight=1, model__subsample=1.0; total time=   5.2s
[CV] END model__colsample_bytree=0.8, model__gamma=0, model__max_depth=6, model__min_child_weight=1, model__subsample=1.0; total time=   5.2s
[CV] END model__colsample_bytree=0.8, model__gamma=0,

{'colsample_bytree': 1.0,
 'gamma': 0,
 'max_depth': 10,
 'min_child_weight': 5,
 'subsample': 1.0}

### Stage B — regularization

Fixes `learning_rate`/`n_estimators` at the same anchor and Stage A's winning tree-complexity params, then searches `reg_lambda`, `reg_alpha`.

Grid size: 4 × 3 = 12 combinations × `CV_FOLDS=3` = **36 fits**.

In [28]:
stage_b_fixed = {**STAGE_A_ANCHOR, **stage_a_best}

stage_b_grid = {
    "reg_lambda": [0.1, 1, 5, 10],
    "reg_alpha": [0, 0.1, 1],
}

search_b = run_grid(stage_b_grid, stage_b_fixed, "Stage B: regularization")
stage_b_best = {k.replace("model__", ""): v for k, v in search_b.best_params_.items()}
stage_b_best

=== Stage B: regularization ===
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END ..........model__reg_alpha=0, model__reg_lambda=0.1; total time=  14.8s
[CV] END ..........model__reg_alpha=0, model__reg_lambda=0.1; total time=  14.7s
[CV] END ..........model__reg_alpha=0, model__reg_lambda=0.1; total time=  14.7s
[CV] END ............model__reg_alpha=0, model__reg_lambda=1; total time=  13.5s
[CV] END ............model__reg_alpha=0, model__reg_lambda=1; total time=  13.8s
[CV] END ............model__reg_alpha=0, model__reg_lambda=1; total time=  13.6s
[CV] END ............model__reg_alpha=0, model__reg_lambda=5; total time=  11.9s
[CV] END ............model__reg_alpha=0, model__reg_lambda=5; total time=  11.8s
[CV] END ............model__reg_alpha=0, model__reg_lambda=5; total time=  11.9s
[CV] END ...........model__reg_alpha=0, model__reg_lambda=10; total time=  11.0s
[CV] END ...........model__reg_alpha=0, model__reg_lambda=10; total time=  11.1s
[CV] END .......

{'reg_alpha': 0, 'reg_lambda': 0.1}

### Stage C — learning rate + estimators

Fixes tree-complexity (Stage A) and regularization (Stage B) winners, then searches the actual `learning_rate` / `n_estimators` trade-off.

Grid size: 4 × 4 = 16 combinations × `CV_FOLDS=3` = **48 fits**. This stage is the slowest per-fit (up to 2000 estimators) despite having the fewest combinations.

In [29]:
stage_c_fixed = {**stage_a_best, **stage_b_best}

stage_c_grid = {
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "n_estimators": [500, 1000, 1500, 2000],
}

search_c = run_grid(stage_c_grid, stage_c_fixed, "Stage C: learning_rate + n_estimators")
stage_c_best = {k.replace("model__", ""): v for k, v in search_c.best_params_.items()}
stage_c_best

=== Stage C: learning_rate + n_estimators ===
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END .model__learning_rate=0.01, model__n_estimators=500; total time=  24.5s
[CV] END .model__learning_rate=0.01, model__n_estimators=500; total time=  24.8s
[CV] END .model__learning_rate=0.01, model__n_estimators=500; total time=  25.0s
[CV] END model__learning_rate=0.01, model__n_estimators=1000; total time=  36.1s
[CV] END model__learning_rate=0.01, model__n_estimators=1000; total time=  37.7s
[CV] END model__learning_rate=0.01, model__n_estimators=1000; total time=  37.0s
[CV] END model__learning_rate=0.01, model__n_estimators=1500; total time=  47.2s
[CV] END model__learning_rate=0.01, model__n_estimators=1500; total time=  48.1s
[CV] END model__learning_rate=0.01, model__n_estimators=1500; total time= 1.2min
[CV] END model__learning_rate=0.01, model__n_estimators=2000; total time=  56.8s
[CV] END model__learning_rate=0.01, model__n_estimators=2000; total time=  58.0s
[C

{'learning_rate': 0.1, 'n_estimators': 2000}

## Final model — combine staged winners and refit on GPU

In [30]:
FINAL_PARAMS = {
    **stage_a_best,
    **stage_b_best,
    **stage_c_best,
}
print("Final tuned XGBoost params:")
FINAL_PARAMS

Final tuned XGBoost params:


{'colsample_bytree': 1.0,
 'gamma': 0,
 'max_depth': 10,
 'min_child_weight': 5,
 'subsample': 1.0,
 'reg_alpha': 0,
 'reg_lambda': 0.1,
 'learning_rate': 0.1,
 'n_estimators': 2000}

In [31]:
# Refit on the full training set with the winning params (device="cuda" - this is
# the GPU artifact, not the deployable one), evaluate on the held-out test set.
final_gpu_pipeline = make_pipeline(**FINAL_PARAMS)
final_gpu_pipeline.fit(X_train, y_train)

train_metrics = evaluate(final_gpu_pipeline, X_train, y_train)
test_metrics = evaluate(final_gpu_pipeline, X_test, y_test)

print("Train metrics:", train_metrics)
print("Test metrics:", test_metrics)
print("Improvement over untuned baseline MAE:", BASELINE_MAE - test_metrics["MAE"])

Train metrics: {'MAE': 1492.5150198518588, 'R2': 0.9821127367599907, 'RMSE': np.float64(2772.8510816421704)}
Test metrics: {'MAE': 1894.9056993477286, 'R2': 0.947451360534117, 'RMSE': np.float64(4645.892696835455)}
Improvement over untuned baseline MAE: 843.8169542458666


## Error analysis

Overall accuracy bands (feed `xgb_deployment_metadata.json`) and segment-level MAE by `make`, `province`, and a `car_age` bucket (feed `dashboard_aggregates.json` so the app can show where the model is weaker).

In [32]:
test_preds = final_gpu_pipeline.predict(X_test)

error_df = X_test.copy()
error_df["actual_price"] = y_test.values
error_df["predicted_price"] = test_preds
error_df["absolute_error"] = (error_df["actual_price"] - error_df["predicted_price"]).abs()
error_df["absolute_percent_error"] = (
    error_df["absolute_error"] / error_df["actual_price"]
) * 100

error_df["car_age_bucket"] = pd.cut(
    error_df["car_age"],
    bins=[-0.01, 3, 6, 10, 15, 100],
    labels=["0-3 yrs", "4-6 yrs", "7-10 yrs", "11-15 yrs", "15+ yrs"]
)

within_1000 = (error_df["absolute_error"] <= 1000).mean() * 100
within_2500 = (error_df["absolute_error"] <= 2500).mean() * 100
within_5000 = (error_df["absolute_error"] <= 5000).mean() * 100
within_10_percent = (error_df["absolute_percent_error"] <= 10).mean() * 100
within_20_percent = (error_df["absolute_percent_error"] <= 20).mean() * 100

accuracy_bands = {
    "within_1000_pct": within_1000,
    "within_2500_pct": within_2500,
    "within_5000_pct": within_5000,
    "within_10_percent_pct": within_10_percent,
    "within_20_percent_pct": within_20_percent,
}
accuracy_bands

{'within_1000_pct': np.float64(48.197023170170326),
 'within_2500_pct': np.float64(79.66297934074518),
 'within_5000_pct': np.float64(93.4255862290234),
 'within_10_percent_pct': np.float64(74.7666941007435),
 'within_20_percent_pct': np.float64(91.34013140457824)}

In [33]:
# Segment-level MAE - which parts of the market is the model weakest on?
# Segments below MIN_SEGMENT_COUNT rows are dropped so small samples don't look noisy.
MIN_SEGMENT_COUNT = 30

def segment_mae(df, col, min_count=MIN_SEGMENT_COUNT):
    grouped = df.groupby(col, observed=True).agg(
        count=("absolute_error", "size"),
        mae=("absolute_error", "mean"),
        median_abs_pct_error=("absolute_percent_error", "median"),
    )
    return grouped[grouped["count"] >= min_count].sort_values("mae", ascending=False)

segment_errors = {
    "make": segment_mae(error_df, "make"),
    "province": segment_mae(error_df, "province"),
    "car_age_bucket": segment_mae(error_df, "car_age_bucket"),
}

for name, table in segment_errors.items():
    print(f"\n--- Worst-performing {name} segments (by MAE) ---")
    display(table.head(10))


--- Worst-performing make segments (by MAE) ---


,count,mae,median_abs_pct_error
make,,,
ferrari,38,32102.430510,7.358793
porsche,278,11195.655031,6.178168
maserati,44,6321.070401,5.818655
tesla,126,4843.089875,4.216832
mercedes-benz,2036,4812.885503,6.024598
land rover,570,4465.538122,5.642173
suzuki,37,3881.708773,20.639780
jaguar,312,3453.964221,5.051580
audi,1472,3254.956989,5.821891



--- Worst-performing province segments (by MAE) ---


,count,mae,median_abs_pct_error
province,,,
bc,9093,2234.405720,5.446369
ab,10578,2148.749779,5.128947
on,22579,1971.208452,5.090819
mb,1984,1811.307733,4.847457
nb,1206,1773.050709,5.403089
qc,18216,1640.554946,4.848965
sk,3066,1411.424749,3.221240
ns,2504,1391.003003,4.517626
pe,260,1356.144201,4.669378



--- Worst-performing car_age_bucket segments (by MAE) ---


,count,mae,median_abs_pct_error
car_age_bucket,,,
15+ yrs,3658,2605.141262,15.510258
4-6 yrs,9176,2327.612863,3.746065
7-10 yrs,43576,1822.925930,4.341065
11-15 yrs,15277,1670.256084,7.620313


## Deployment export

- `xgb_gpu_pipeline.joblib` — the GPU pipeline as-is (training artifact)
- `xgb_deployment_pipeline.joblib` — CPU re-fit of the identical params, so inference never tries to grab CUDA on a CPU-only host (e.g. Streamlit Community Cloud)
- `xgb_deployment_metadata.json` — expected columns, params, metrics, accuracy bands, artifact size, exact library versions

In [34]:
# CPU re-fit: identical params, device="cpu" instead of "cuda".
# Re-fit rather than converting the GPU booster, so the deployment artifact
# is trained the same way it will run in production.
DEPLOYMENT_PARAMS = {**FINAL_PARAMS, "device": "cpu"}

deployment_pipeline = make_pipeline(**DEPLOYMENT_PARAMS)
deployment_pipeline.fit(X_train, y_train)

deployment_test_metrics = evaluate(deployment_pipeline, X_test, y_test)
print("CPU deployment pipeline test metrics:", deployment_test_metrics)

CPU deployment pipeline test metrics: {'MAE': 1893.0420467787462, 'R2': 0.9461873308450931, 'RMSE': np.float64(4701.437898169292)}


In [35]:
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(final_gpu_pipeline, "../models/xgb_gpu_pipeline.joblib")
joblib.dump(deployment_pipeline, "../models/xgb_deployment_pipeline.joblib")

deployment_artifact_size_bytes = Path("../models/xgb_deployment_pipeline.joblib").stat().st_size
deployment_artifact_size_mb = deployment_artifact_size_bytes / (1024 ** 2)
print(f"Deployment pipeline size: {deployment_artifact_size_mb:.2f} MB")

Deployment pipeline size: 17.27 MB


In [36]:
import sys
import sklearn
import xgboost
import pyarrow

library_versions = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__,
    "pyarrow": pyarrow.__version__,
}

metadata = {
    "expected_columns": X_train.columns.tolist(),
    "final_params": FINAL_PARAMS,
    "test_metrics": deployment_test_metrics,
    "accuracy_bands": accuracy_bands,
    "artifact_size_mb": round(deployment_artifact_size_mb, 2),
    "library_versions": library_versions,
}

with open("../models/xgb_deployment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

metadata

{'expected_columns': ['miles',
  'year',
  'make',
  'body_type',
  'vehicle_type',
  'drivetrain',
  'transmission',
  'fuel_type',
  'engine_size',
  'engine_block',
  'city',
  'province',
  'miles_was_zero',
  'car_age',
  'miles_per_year',
  'model_trim',
  'region'],
 'final_params': {'colsample_bytree': 1.0,
  'gamma': 0,
  'max_depth': 10,
  'min_child_weight': 5,
  'subsample': 1.0,
  'reg_alpha': 0,
  'reg_lambda': 0.1,
  'learning_rate': 0.1,
  'n_estimators': 2000},
 'test_metrics': {'MAE': 1893.0420467787462,
  'R2': 0.9461873308450931,
  'RMSE': np.float64(4701.437898169292)},
 'accuracy_bands': {'within_1000_pct': np.float64(48.197023170170326),
  'within_2500_pct': np.float64(79.66297934074518),
  'within_5000_pct': np.float64(93.4255862290234),
  'within_10_percent_pct': np.float64(74.7666941007435),
  'within_20_percent_pct': np.float64(91.34013140457824)},
 'artifact_size_mb': 17.27,
 'library_versions': {'python': '3.14.5',
  'pandas': '3.0.2',
  'numpy': '2.4.4',
 

## Load-back verification

Reload the deployment artifact and metadata from disk (not from the in-memory objects above) and confirm they behave exactly as expected — this is the same path the Streamlit app will take.

In [37]:
# Reload from disk (not the in-memory pipeline) and confirm predictions match exactly
reloaded_pipeline = joblib.load("../models/xgb_deployment_pipeline.joblib")

original_preds = deployment_pipeline.predict(X_test)
reloaded_preds = reloaded_pipeline.predict(X_test)

assert np.allclose(original_preds, reloaded_preds, rtol=1e-6, atol=1e-6), \
    "Reloaded pipeline predictions do not match the in-memory pipeline!"
print("Reload verification passed: predictions match.")
print("Max absolute difference:", np.max(np.abs(original_preds - reloaded_preds)))

with open("../models/xgb_deployment_metadata.json") as f:
    reloaded_metadata = json.load(f)

assert reloaded_metadata["expected_columns"] == X_train.columns.tolist(), \
    "Reloaded expected_columns do not match X_train.columns!"
print("Metadata reload verification passed: expected_columns match.")

Reload verification passed: predictions match.
Max absolute difference: 0.0
Metadata reload verification passed: expected_columns match.


## Dashboard aggregates export

Precomputed, small JSON so the Streamlit app's Explore/Model-Info pages never have to load the full 390k-row dataset into memory.

In [38]:
DASHBOARD_MIN_SEGMENT_COUNT = 30

price_hist_counts, price_hist_edges = np.histogram(
    df_model.loc[df_model["price"] <= df_model["price"].quantile(0.99), "price"],
    bins=40
)

def price_by_segment(df, col, min_count=DASHBOARD_MIN_SEGMENT_COUNT):
    grouped = df.groupby(col, observed=True)["price"].agg(count="size", median_price="median", mean_price="mean")
    return grouped[grouped["count"] >= min_count].sort_values("count", ascending=False)

def price_segment_to_dict(table):
    return {
        str(idx): {
            "count": int(row["count"]),
            "median_price": round(float(row["median_price"]), 2),
            "mean_price": round(float(row["mean_price"]), 2),
        }
        for idx, row in table.iterrows()
    }

def error_segment_to_dict(table):
    return {
        str(idx): {
            "count": int(row["count"]),
            "mae": round(float(row["mae"]), 2),
            "median_abs_pct_error": round(float(row["median_abs_pct_error"]), 2),
        }
        for idx, row in table.iterrows()
    }

dashboard_aggregates = {
    "price_distribution": {
        "bin_edges": price_hist_edges.tolist(),
        "counts": price_hist_counts.tolist(),
    },
    "overall": {
        "count": int(len(df_model)),
        "median_price": round(float(df_model["price"].median()), 2),
        "mean_price": round(float(df_model["price"].mean()), 2),
    },
    "price_by_make": price_segment_to_dict(price_by_segment(df_model, "make").head(20)),
    "price_by_province": price_segment_to_dict(price_by_segment(df_model, "province")),
    "price_by_body_type": price_segment_to_dict(price_by_segment(df_model, "body_type")),
    "error_by_make": error_segment_to_dict(segment_errors["make"].head(20)),
    "error_by_province": error_segment_to_dict(segment_errors["province"]),
    "error_by_car_age_bucket": error_segment_to_dict(segment_errors["car_age_bucket"]),
}

with open("../models/dashboard_aggregates.json", "w") as f:
    json.dump(dashboard_aggregates, f, indent=2)

print("Dashboard aggregates saved:", list(dashboard_aggregates.keys()))

Dashboard aggregates saved: ['price_distribution', 'overall', 'price_by_make', 'price_by_province', 'price_by_body_type', 'error_by_make', 'error_by_province', 'error_by_car_age_bucket']


## Form options export

Precomputed dropdown options for the Streamlit app, so the app never loads the full dataset just to populate its input widgets. Includes the low-cardinality category lists, the cascading `make → model_trim` and `province → city` maps (frequency-filtered to mirror the encoders' `min_frequency`), and numeric ranges for widget bounds. Saved as `models/form_options.json`.

In [ ]:
"""
Form options export for the Streamlit app's input widgets.

Built from df_model (post-feature-engineering, pre-split) so the dropdown options
match exactly what the fitted OneHotEncoders learned. The high-cardinality
frequency cutoffs mirror the encoders' min_frequency settings (model_trim=50,
city=100 above), so every option shown is a value the model actually encodes
rather than one collapsed into the infrequent bucket.
"""
# Values the SimpleImputer injects for NaNs, plus junk - never show these.
SENTINELS = {"missing", "unknown", "nan", "none", "", "unknown unknown"}

# The 8 columns that go through the low_cardinality one-hot transformer.
LOW_CARD_COLS = [
    "make", "body_type", "vehicle_type", "drivetrain",
    "transmission", "fuel_type", "engine_block", "province",
]

# Mirror the encoders' min_frequency so shown options == encoded options.
MODEL_TRIM_MIN_FREQ = 50
CITY_MIN_FREQ = 100


def _clean_series(series):
    s = series.dropna().astype(str).str.strip()
    return s[~s.str.lower().isin(SENTINELS)]


def _low_card_options(col):
    return sorted(_clean_series(df_model[col]).unique().tolist())


def _frequent_values(col, min_freq):
    vc = _clean_series(df_model[col]).value_counts()
    return set(vc[vc >= min_freq].index)


def _cascading_map(key_col, val_col, allowed_vals):
    sub = df_model[[key_col, val_col]].dropna().copy()
    sub[key_col] = sub[key_col].astype(str).str.strip()
    sub[val_col] = sub[val_col].astype(str).str.strip()
    sub = sub[~sub[key_col].str.lower().isin(SENTINELS)]
    sub = sub[sub[val_col].isin(allowed_vals)]
    out = {}
    for key, grp in sub.groupby(key_col):
        vals = sorted(grp[val_col].unique().tolist())
        if vals:
            out[key] = vals
    return out


def _numeric_range(col):
    s = df_model[col].dropna()
    return {
        "min": round(float(s.min()), 2),
        "p01": round(float(s.quantile(0.01)), 2),
        "median": round(float(s.median()), 2),
        "p99": round(float(s.quantile(0.99)), 2),
        "max": round(float(s.max()), 2),
    }


freq_model_trims = _frequent_values("model_trim", MODEL_TRIM_MIN_FREQ)
freq_cities = _frequent_values("city", CITY_MIN_FREQ)

form_options = {
    "low_cardinality": {c: _low_card_options(c) for c in LOW_CARD_COLS},
    "make_to_model_trims": _cascading_map("make", "model_trim", freq_model_trims),
    "province_to_cities": _cascading_map("province", "city", freq_cities),
    "numeric_ranges": {c: _numeric_range(c) for c in ["year", "miles", "engine_size"]},
}

with open("../models/form_options.json", "w") as f:
    json.dump(form_options, f, indent=2)

print("Form options saved:", list(form_options.keys()))
print("low_cardinality counts:", {k: len(v) for k, v in form_options["low_cardinality"].items()})
print("makes with model_trims:", len(form_options["make_to_model_trims"]),
      "| frequent model_trims:", len(freq_model_trims))
print("provinces with cities:", len(form_options["province_to_cities"]),
      "| frequent cities:", len(freq_cities))

## Conclusion

**Objective.** This notebook trained, tuned, and exported the production XGBoost model for the used-car price estimator. The model choice was driven by a deployment constraint rather than accuracy: the earlier RandomForest reached usable performance but serialized to ~4.7 GB, which is impractical to load into a memory-limited serverless container. XGBoost was selected because it serializes to a fraction of that size at comparable accuracy.

**Tuning.** Hyperparameters were tuned with a staged GridSearch (coordinate descent) rather than a single combinatorial grid — tree complexity, then regularization, then the learning-rate / estimator trade-off, with each stage's winners fixed for the next. Staging kept every individual grid small enough to search exhaustively while avoiding the combinatorial blow-up of tuning all parameters simultaneously.

Regularization contributed nothing measurable. Across `reg_lambda` from 0 to 0.5, test MAE varied by under $10 — within noise — and values above 1 strictly degraded performance. The train/test gap on this dataset is not the kind that L2 penalties correct.

**Where the ceiling is.** A learning-curve sweep showed test MAE flattening well before the full dataset was used: the final 10% of rows bought roughly $10 of MAE. Train MAE simultaneously *rose* as data grew (from ~$827 at 10% to ~$1,493 at 100%) as memorization became impossible, with the two curves converging toward a persistent ~$400 gap. That gap is not reducible with the current feature set — it reflects price variance that isn't captured in the available columns, such as vehicle condition, accident history, and seller motivation. More data will not help; additional features would.

**Where the model is weak.** Absolute error grows with price, from ~$1,310 MAE in the cheapest decile to ~$5,095 in the most expensive. Relative error runs the opposite direction: roughly 18% in the bottom decile against roughly 8% in the top. The model is proportionally *worst on cheap cars*, a pattern the dollar-denominated metric conceals. The top price band also shows an RMSE/MAE ratio near 2.5 — the widest in the table — indicating a small number of extreme errors consistent with data-quality artifacts at the upper end of the price range.

**Chosen trade-off.** Sweeps showed test MAE continuing to improve monotonically with deeper trees and more estimators, reaching ~$1,634 at `max_depth=16` / `n_estimators=4000` without turning over. Those gains were not taken. The improvement over the deployed configuration is roughly $260 — about 1.3% of a $20,000 vehicle — and comes at substantial artifact growth, directly opposing the constraint that motivated moving off RandomForest in the first place. The deployed configuration favours a smaller artifact and accepts the marginally higher error.

**Artifacts produced.**
- `xgb_gpu_pipeline.joblib` — the GPU training artifact
- `xgb_deployment_pipeline.joblib` — a CPU re-fit of identical parameters, so inference never attempts to acquire CUDA on a CPU-only host
- `xgb_deployment_metadata.json` — expected column schema and order, final parameters, test metrics, accuracy bands, artifact size, and exact library versions for dependency pinning
- `dashboard_aggregates.json` — precomputed segment statistics and price distributions, so downstream consumers never need to load the full dataset

Load-back verification confirmed the deployment artifact reproduces predictions from disk and that the persisted column schema matches the training set.